In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os
import torch

In [ ]:
# Add project root to path so we can import vae, dit, utils, etc.
# In notebooks, we use getcwd() or hardcode the path
project_root = "/Users/siddarthnilolkundursatish/Desktop/NYU/Courses/Computer_Vision/PhysVideoGenerator"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"sys.path[0]: {sys.path[0]}")

Project root: /Users/siddarthnilolkundursatish/Desktop/NYU/Courses/Computer_Vision/PhysVideoGenerator
sys.path[0]: /Users/siddarthnilolkundursatish/Desktop/NYU/Courses/Computer_Vision/PhysVideoGenerator


In [ ]:
from vae import vae_encoder_decoder
from vjepa2 import vjepa2_encoder

/Users/siddarthnilolkundursatish/Desktop/NYU/Courses/Computer_Vision/PhysVideoGenerator/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
vae_encoding_output_folder = os.path.join(project_root, "data/encoded_videos/vae_encoded")
os.makedirs(vae_encoding_output_folder, exist_ok=True)

In [ ]:
vae_encode_dict = vae_encoder_decoder.encode_video(
    npz_folder_path=os.path.join(project_root, "data/clean_video_npz"),
    output_folder_path=vae_encoding_output_folder,
    dtype=torch.float16,
    device="cpu"
)

/Users/siddarthnilolkundursatish/Desktop/NYU/Courses/Computer_Vision/PhysVideoGenerator/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Encoding videos using VAE::   0%|          | 0/1000 [00:00<?, ?it/s]

In [ ]:
vjepa2_encoding_output_folder = os.path.join(project_root, "data/encoded_videos/vjepa2_encoded")
os.makedirs(vjepa2_encoding_output_folder, exist_ok=True)

In [ ]:
from transformers import AutoVideoProcessor, AutoModel

In [ ]:
def forward_vjepa_video(cleaned_video_npz, model_hf, hf_transform):
    # Run a sample inference with VJEPA
    with torch.inference_mode():
        # Read and pre-process the image
        video = torch.from_numpy(cleaned_video_npz).permute(0, 3, 1, 2)  # T x C x H x W
        x_hf = hf_transform(video, return_tensors="pt")["pixel_values_videos"].to("cuda")
        # Extract the patch-wise features from the last layer
        out_patch_features_hf = model_hf.get_vision_features(x_hf)

    return out_patch_features_hf

In [ ]:
# HuggingFace model repo name
hf_model_name = (
    "facebook/vjepa2-vitg-fpc64-384"  # Replace with your favored model, e.g. facebook/vjepa2-vitg-fpc64-384
)

# Initialize the HuggingFace model, load pretrained weights
model_hf = AutoModel.from_pretrained(hf_model_name)
model_hf.cuda().eval()

# Build HuggingFace preprocessing transform
hf_transform = AutoVideoProcessor.from_pretrained(hf_model_name)
img_size = hf_transform.crop_size["height"]  # E.g. 384, 256, etc.

# Inference on video
out_patch_features_hf = vjepa2_encoder.forward_vjepa_video(model_hf, hf_transform)

print(
    f"""
    Inference results on video:
    HuggingFace output shape: {out_patch_features_hf.shape}
    """
)